After this cell finishes, restart the Colab runtime before running imports:

Runtime → Restart session

Colab may print dependency warnings for preinstalled packages like gradio, jax, opencv, shap, or rasterio. Those are expected and can be ignored as long as the verification cell below imports successfully.


In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.1 \
  pyarrow==17.0.0 \
  fsspec==2024.6.1 \
  pillow==11.3.0 \
  requests==2.32.4 \
  jedi==0.19.1 \
  transformers==4.44.2 \
  accelerate==0.33.0 \
  bitsandbytes==0.43.3 \
  datasets==2.21.0 \
  matplotlib==3.9.2 \
  seaborn==0.13.2 \
  kaggle==1.6.17 \
  huggingface_hub==0.25.2 \
  wandb==0.17.5


In [ ]:
import transformers
import datasets
import sklearn
import pandas
import numpy
import scipy
import pyarrow
import PIL
import matplotlib
import huggingface_hub
import fsspec

print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)
print("numpy", numpy.__version__)
print("scipy", scipy.__version__)
print("pyarrow", pyarrow.__version__)
print("pillow", PIL.__version__)
print("matplotlib", matplotlib.__version__)
print("huggingface_hub", huggingface_hub.__version__)
print("fsspec", fsspec.__version__)
print("Environment ready.")


In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
target = kaggle_dir / "kaggle.json"

if not target.exists():
    print("Upload your Kaggle API token file named kaggle.json.")
    if files is None:
        raise RuntimeError("google.colab.files is unavailable. Place kaggle.json at ~/.kaggle/kaggle.json manually.")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Expected an uploaded file named kaggle.json")
    shutil.move("kaggle.json", target)

if target.exists():
    target.chmod(0o600)

print("Kaggle token exists:", target.exists())
print("Kaggle token path:", target)


In [ ]:
from pathlib import Path

kaggle_path = Path.home() / ".kaggle" / "kaggle.json"
print("Kaggle token exists:", kaggle_path.exists())


In [ ]:
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
# Persistent Colab handoff paths for H9-H16.
# Google Drive is the source of truth across separate Colab notebooks/runtimes.
from pathlib import Path

USE_GOOGLE_DRIVE = True

try:
    from google.colab import drive
    if USE_GOOGLE_DRIVE:
        drive.mount("/content/drive")
        NOTEBOOKS_ROOT = Path("/content/drive/MyDrive/GemScan/notebooks")
    else:
        NOTEBOOKS_ROOT = Path("/content/GemScan/notebooks")
except ModuleNotFoundError:
    # Local fallback for VS Code/Jupyter outside Colab.
    NOTEBOOKS_ROOT = Path("notebooks")

DATA_DIR = NOTEBOOKS_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SCRUBBED_DIR = DATA_DIR / "scrubbed"
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
FIXTURES_DIR = DATA_DIR / "fixtures"

for path in [RAW_DIR, PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("NOTEBOOKS_ROOT", NOTEBOOKS_ROOT)
print("RAW_DIR", RAW_DIR)
print("PROCESSED_DIR", PROCESSED_DIR)
print("SCRUBBED_DIR", SCRUBBED_DIR)
print("RESULTS_DIR", RESULTS_DIR)
print("FIXTURES_DIR", FIXTURES_DIR)


# H9.1 PULL UCI SMS SPAM COLLECTION


In [ ]:
from pathlib import Path
import zipfile
import urllib.request
import pandas as pd

# Uses RAW_DIR and PROCESSED_DIR from the persistent Drive path setup cell.
uci_raw_dir = RAW_DIR / "uci_sms_spam"
uci_raw_dir.mkdir(parents=True, exist_ok=True)

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
zip_path = uci_raw_dir / "smsspamcollection.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(uci_raw_dir)

sms_path = uci_raw_dir / "SMSSpamCollection"

df = pd.read_csv(
    sms_path,
    sep="	",
    header=None,
    names=["label", "text"],
    encoding="latin-1",
)

df["id"] = [f"uci-{i:05d}" for i in range(len(df))]
df["source"] = "uci_sms_spam_collection"
df["language"] = "en"
df["split"] = "unassigned"

df = df[["id", "source", "text", "label", "language", "split"]]

out_path = PROCESSED_DIR / "uci_sms_clean.csv"
df.to_csv(out_path, index=False)

print(df.shape)
print(df["label"].value_counts())
print("saved", out_path)
df.head()


# H9.2 Kaggle SMS-Spam datasets

In [ ]:
from pathlib import Path
import zipfile
import subprocess

# Uses RAW_DIR from the persistent Drive path setup cell.
kaggle_raw_dir = RAW_DIR / "kaggle"
kaggle_raw_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    "abhishek14398/sms-spam-collection",
    "vishakhdapat/sms-spam-detection-dataset",
]

for dataset in datasets:
    owner, name = dataset.split("/")
    out_dir = kaggle_raw_dir / name
    out_dir.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "kaggle", "datasets", "download",
            "-d", dataset,
            "-p", str(out_dir),
            "--force",
        ],
        check=True,
    )

    for zip_path in out_dir.glob("*.zip"):
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(out_dir)

    print("Downloaded:", dataset)
    print("Files:", [p.name for p in out_dir.iterdir()])


In [ ]:
import pandas as pd

for csv_path in (RAW_DIR / "kaggle").glob("**/*.csv"):
    print()
    print(csv_path)
    try:
        preview = pd.read_csv(csv_path, encoding="utf-8")
    except UnicodeDecodeError:
        preview = pd.read_csv(csv_path, encoding="latin-1")

    print(preview.shape)
    print(preview.columns.tolist())
    display(preview.head())


# H9.3 Multilingual scam corpus

In [ ]:
from datasets import load_dataset
import pandas as pd
from pathlib import Path

# Uses RAW_DIR and PROCESSED_DIR from the persistent Drive path setup cell.
multilingual_raw_dir = RAW_DIR / "multilingual"
multilingual_raw_dir.mkdir(parents=True, exist_ok=True)

dataset_id = "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset"

ds = load_dataset(dataset_id, split="train")
df = ds.to_pandas()

raw_snapshot_path = multilingual_raw_dir / "sms_spam_multilingual_snapshot.csv"
df.to_csv(raw_snapshot_path, index=False)

print(df.shape)
print(df.columns.tolist())
print("saved raw snapshot", raw_snapshot_path)
display(df.head())


# H9.5 - PII Scrub and Build Evaluation Corpora

This step scrubs every pulled CSV, then writes separate handoff datasets:

- `processed/h9_local_sms_corpus_scrubbed.csv`: clean H10 zero-shot baseline corpus, limited to English UCI/Kaggle SMS `ham/spam` rows.
- `processed/h9_zero_shot_baseline_corpus_scrubbed.csv`: same clean H10 corpus under an explicit name.
- `processed/h9_local_all_text_corpus_scrubbed.csv`: audit-only mixed corpus containing every normalized text dataset. Do not use this for H10.
- `scrubbed/h9_multilingual_selected_languages__scrubbed.csv`: H13 multilingual corpus.

The H10 corpus intentionally excludes multilingual rows and synthetic hard cases so zero-shot metrics answer one question: how Gemma performs on classic SMS scam/spam detection before prompt tuning, thresholding, or fine-tuning.


In [ ]:
import json
import re
import shutil
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Uses DATA_DIR / RAW_DIR / PROCESSED_DIR / SCRUBBED_DIR / RESULTS_DIR / FIXTURES_DIR
# from the persistent Drive path setup cell. CSVs written here survive separate Colab notebooks.
for directory in [RAW_DIR, PROCESSED_DIR, SCRUBBED_DIR, RESULTS_DIR, FIXTURES_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Copy versioned synthetic fixtures from the repo into Drive so H11/H14/H15 can read them in separate runtimes.
repo_fixture_candidates = [
    Path("notebooks/data/fixtures/hard_cases_v0.csv"),
    Path("GemScan/notebooks/data/fixtures/hard_cases_v0.csv"),
]
for candidate in repo_fixture_candidates:
    if candidate.exists():
        shutil.copy2(candidate, FIXTURES_DIR / candidate.name)
        print("copied fixture to Drive", FIXTURES_DIR / candidate.name)
        break

PII_PATTERNS = {
    "email": re.compile(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", re.IGNORECASE),
    "url": re.compile(r"\b(?:https?://|www\.)\S+", re.IGNORECASE),
    "credit_or_long_payment_number": re.compile(r"(?<!\w)(?:\d[ -]?){13,19}(?!\w)"),
    "phone_number": re.compile(r"(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)"),
    "crypto_wallet": re.compile(r"\b(?:0x[a-fA-F0-9]{40}|bc1[a-zA-HJ-NP-Z0-9]{25,59}|[13][a-km-zA-HJ-NP-Z1-9]{25,34})\b"),
}

REPLACEMENTS = {
    "email": "[EMAIL_REDACTED]",
    "url": "[URL_REDACTED]",
    "credit_or_long_payment_number": "[PAYMENT_NUMBER_REDACTED]",
    "phone_number": "[PHONE_REDACTED]",
    "crypto_wallet": "[WALLET_REDACTED]",
}

TEXT_COLUMN_CANDIDATES = ["text", "message", "sms", "v2", "Text", "Message", "SMS", "body", "Body"]
LABEL_COLUMN_CANDIDATES = ["label", "labels", "category", "v1", "Label", "Category", "class", "Class", "expected_label"]
BASELINE_LABELS = {"ham", "spam"}
H10_BASELINE_SOURCE_MARKERS = [
    "uci_sms_clean",
    "uci_sms_spam_collection",
    "sms-spam-collection",
    "sms-spam-detection-dataset",
]
EXCLUDED_H10_SOURCE_MARKERS = ["multilingual", "hard_cases", "fixture"]


def read_csv_flexible(path: Path) -> pd.DataFrame:
    for encoding in ["utf-8", "latin-1"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="latin-1", errors="replace")


def redact_text(value, counts):
    if pd.isna(value):
        return value
    text = str(value)
    for key, pattern in PII_PATTERNS.items():
        text, n = pattern.subn(REPLACEMENTS[key], text)
        counts[key] += n
    text = re.sub(r"\s+", " ", text).strip()
    return text


def scrub_dataframe(frame: pd.DataFrame):
    counts = {key: 0 for key in PII_PATTERNS}
    scrubbed = frame.copy()
    object_columns = scrubbed.select_dtypes(include=["object", "string"]).columns
    for column in object_columns:
        scrubbed[column] = scrubbed[column].map(lambda value: redact_text(value, counts))
    return scrubbed, counts


def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


def normalize_label(value: object) -> str:
    label = str(value).lower().strip()
    label = re.sub(r"\s+", "_", label)
    return {
        "0": "ham",
        "1": "spam",
        "legitimate": "ham",
        "benign": "ham",
        "safe": "ham",
        "not_spam": "ham",
        "non_spam": "ham",
        "normal": "ham",
        "phishing": "spam",
        "fraud": "spam",
        "scam": "spam",
        "malicious": "spam",
    }.get(label, label)


def normalize_language(value: object) -> str:
    if pd.isna(value):
        return "en"
    language = str(value).strip()
    if language.lower() in {"", "nan", "none", "english"}:
        return "en"
    return language


def normalize_for_combined(frame: pd.DataFrame, source_name: str) -> pd.DataFrame | None:
    text_col = first_existing(frame.columns, TEXT_COLUMN_CANDIDATES)
    label_col = first_existing(frame.columns, LABEL_COLUMN_CANDIDATES)
    if text_col is None or label_col is None:
        return None
    normalized = pd.DataFrame(
        {
            "id": [f"{source_name}-{i:06d}" for i in range(len(frame))],
            "source": source_name,
            "text": frame[text_col].astype(str),
            "label": frame[label_col].map(normalize_label),
            "language": frame["language"].map(normalize_language) if "language" in frame.columns else "en",
            "split": frame["split"].astype(str).str.lower().str.strip() if "split" in frame.columns else "unassigned",
        }
    )
    normalized["text"] = normalized["text"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
    normalized = normalized[normalized["text"].ne("")]
    normalized = normalized[~normalized["text"].str.lower().isin({"nan", "none"})]
    return normalized


def safe_output_name(path: Path) -> str:
    try:
        relative = path.relative_to(NOTEBOOKS_ROOT)
    except ValueError:
        relative = path
    parts = [part for part in relative.with_suffix("").parts if part not in {"data", "raw", "processed", "fixtures", "scrubbed"}]
    return "__".join(parts).replace(" ", "_")


def is_h10_source(source: str) -> bool:
    lowered = source.lower()
    if any(marker in lowered for marker in EXCLUDED_H10_SOURCE_MARKERS):
        return False
    return any(marker in lowered for marker in H10_BASELINE_SOURCE_MARKERS)


def add_text_key(frame: pd.DataFrame) -> pd.DataFrame:
    keyed = frame.copy()
    keyed["text_key"] = keyed["text"].astype(str).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    return keyed


def dedupe_for_eval(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    keyed = add_text_key(frame)
    label_counts = keyed.groupby("text_key")["label"].nunique()
    conflicting_keys = set(label_counts[label_counts > 1].index)
    conflicts = keyed[keyed["text_key"].isin(conflicting_keys)].copy()
    deduped = keyed[~keyed["text_key"].isin(conflicting_keys)].copy()
    deduped = deduped.sort_values(["source", "id"]).drop_duplicates(subset=["text_key"], keep="first")
    return deduped.drop(columns=["text_key"]).reset_index(drop=True), conflicts.drop(columns=["text_key"]).reset_index(drop=True)




def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "_No rows._"
    markdown_frame = frame.fillna("").astype(str)
    columns = list(markdown_frame.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for _, row in markdown_frame.iterrows():
        values = [str(row[column]).replace("|", "\\|") for column in columns]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)


def assign_stratified_splits(frame: pd.DataFrame, seed: int = 0) -> pd.DataFrame:
    split_ready = frame.copy().reset_index(drop=True)
    if split_ready["label"].nunique() < 2 or split_ready["label"].value_counts().min() < 3:
        split_ready["split"] = "test"
        return split_ready

    train_idx, temp_idx = train_test_split(
        split_ready.index,
        test_size=0.30,
        random_state=seed,
        stratify=split_ready["label"],
    )
    temp = split_ready.loc[temp_idx]
    if temp["label"].nunique() < 2 or temp["label"].value_counts().min() < 2:
        split_ready["split"] = "train"
        split_ready.loc[temp_idx, "split"] = "test"
        return split_ready

    val_idx, test_idx = train_test_split(
        temp.index,
        test_size=0.50,
        random_state=seed,
        stratify=temp["label"],
    )
    split_ready["split"] = "train"
    split_ready.loc[val_idx, "split"] = "val"
    split_ready.loc[test_idx, "split"] = "test"
    return split_ready


summary_rows = []
combined_frames = []
input_paths = []

for base_dir in [RAW_DIR, PROCESSED_DIR, FIXTURES_DIR]:
    if base_dir.exists():
        input_paths.extend(path for path in base_dir.glob("**/*.csv") if "scrubbed" not in path.name.lower())

for csv_path in sorted(set(input_paths)):
    frame = read_csv_flexible(csv_path)
    scrubbed, counts = scrub_dataframe(frame)
    output_name = f"{safe_output_name(csv_path)}__scrubbed.csv"
    output_path = SCRUBBED_DIR / output_name
    scrubbed.to_csv(output_path, index=False)

    source_name = safe_output_name(csv_path)
    normalized = normalize_for_combined(scrubbed, source_name)
    if normalized is not None:
        combined_frames.append(normalized)

    summary_rows.append(
        {
            "input_path": str(csv_path),
            "output_path": str(output_path),
            "source_name": source_name,
            "h10_candidate_source": is_h10_source(source_name),
            "rows": len(frame),
            **counts,
        }
    )

# If the H9.3 multilingual Hugging Face dataframe is still in memory, create the selected-language seed now.
hf_multilingual = globals().get("df")
if isinstance(hf_multilingual, pd.DataFrame):
    language_columns = {
        "en": "text",
        "es": "text_es",
        "hi": "text_hi",
        "zh-Hans": "text_zh",
        "ja": "text_ja",
    }
    if "labels" in hf_multilingual.columns and all(column in hf_multilingual.columns for column in language_columns.values()):
        multilingual_rows = []
        for row_index, row in hf_multilingual.iterrows():
            for language, column in language_columns.items():
                value = row[column]
                if pd.isna(value) or not str(value).strip():
                    continue
                multilingual_rows.append(
                    {
                        "id": f"local-multilingual-{row_index:05d}-{language}",
                        "source": "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset",
                        "text": str(value),
                        "label": normalize_label(row["labels"]),
                        "language": language,
                        "split": "unassigned",
                    }
                )
        multilingual_frame = pd.DataFrame(multilingual_rows)
        multilingual_scrubbed, counts = scrub_dataframe(multilingual_frame)
        multilingual_path = SCRUBBED_DIR / "h9_multilingual_selected_languages__scrubbed.csv"
        multilingual_scrubbed.to_csv(multilingual_path, index=False)
        combined_frames.append(multilingual_scrubbed)
        summary_rows.append(
            {
                "input_path": "in_memory_hf_multilingual_selected_languages",
                "output_path": str(multilingual_path),
                "source_name": "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset",
                "h10_candidate_source": False,
                "rows": len(multilingual_frame),
                **counts,
            }
        )

summary = pd.DataFrame(summary_rows)
summary_path = RESULTS_DIR / "h9_pii_scrub_summary.csv"
summary.to_csv(summary_path, index=False)

if combined_frames:
    combined = pd.concat(combined_frames, ignore_index=True)
    combined = combined.drop_duplicates(subset=["source", "text", "label", "language"])
    all_corpus_path = PROCESSED_DIR / "h9_local_all_text_corpus_scrubbed.csv"
    combined.to_csv(all_corpus_path, index=False)

    h10_candidates = combined[
        combined["source"].map(is_h10_source)
        & combined["language"].eq("en")
        & combined["label"].isin(BASELINE_LABELS)
    ].copy()
    h10_candidates, conflicting_duplicates = dedupe_for_eval(h10_candidates)
    h10_candidates = assign_stratified_splits(h10_candidates, seed=0)

    explicit_h10_path = PROCESSED_DIR / "h9_zero_shot_baseline_corpus_scrubbed.csv"
    compatibility_h10_path = PROCESSED_DIR / "h9_local_sms_corpus_scrubbed.csv"
    h10_candidates.to_csv(explicit_h10_path, index=False)
    h10_candidates.to_csv(compatibility_h10_path, index=False)

    conflict_path = RESULTS_DIR / "h9_zero_shot_conflicting_duplicates.csv"
    conflicting_duplicates.to_csv(conflict_path, index=False)

    manifest_rows = []
    for source, frame in combined.groupby("source"):
        manifest_rows.append(
            {
                "source": source,
                "rows": len(frame),
                "languages": ",".join(sorted(frame["language"].dropna().astype(str).unique())),
                "labels": ",".join(sorted(frame["label"].dropna().astype(str).unique())),
                "used_for_h10_zero_shot": is_h10_source(source),
                "h10_note": "classic English SMS ham/spam source" if is_h10_source(source) else "reserved for non-H10 robustness, prompt, or audit work",
            }
        )
    manifest = pd.DataFrame(manifest_rows).sort_values(["used_for_h10_zero_shot", "source"], ascending=[False, True])
    manifest_path = RESULTS_DIR / "h9_dataset_manifest.csv"
    manifest.to_csv(manifest_path, index=False)

    report_path = RESULTS_DIR / "h9_zero_shot_dataset_report.md"
    report_lines = [
        "# H9 Zero-shot Baseline Dataset Report",
        "",
        "H10 should use the clean English UCI/Kaggle SMS corpus only. Multilingual and synthetic hard-case data are intentionally excluded from this handoff.",
        "",
        f"Explicit H10 corpus: `{explicit_h10_path}`",
        f"Compatibility H10 corpus: `{compatibility_h10_path}`",
        f"Audit-only mixed corpus: `{all_corpus_path}`",
        f"Dataset manifest: `{manifest_path}`",
        f"Conflicting duplicate rows removed: `{len(conflicting_duplicates)}` (`{conflict_path}`)",
        "",
        "## H10 Corpus Shape",
        "",
        dataframe_to_markdown(h10_candidates.groupby(["split", "label"]).size().reset_index(name="rows")),
        "",
        "## Source Manifest",
        "",
        dataframe_to_markdown(manifest),
        "",
    ]
    report_path.write_text("\n".join(report_lines))

    print("H10 zero-shot corpus:", h10_candidates.shape, explicit_h10_path)
    print("Compatibility H10 corpus:", compatibility_h10_path)
    print("Audit-only mixed corpus:", combined.shape, all_corpus_path)
    print("Conflicting duplicate rows removed from H10:", len(conflicting_duplicates), conflict_path)
    print("Dataset manifest:", manifest_path)
    print("Dataset report:", report_path)
    display(h10_candidates.groupby(["split", "label"]).size().reset_index(name="rows"))
    display(manifest)
else:
    print("No normalized text/label corpora found for combined output.")

print("Scrub summary:", summary_path)
display(summary)


# H9.6 - Image Zero-shot Corpus

Build the reusable image benchmark handoff for H10.5 from the local ham/spam image folders.

Ground-truth mapping:
- `personal_image_ham` -> `safe`
- `personal_image_spam` -> `spam`
- `spam_archive_jmlr` -> `spam`

The corpus keeps a relative path so downstream notebooks can load images, but H10.5 prediction and metrics artifacts use only privacy-aware stable image IDs plus source-folder metadata.


In [ ]:
from pathlib import Path
import hashlib
import json

import pandas as pd
from PIL import Image, ImageSequence, ImageOps
from sklearn.model_selection import train_test_split

IMAGE_DATASET_SCHEMA_VERSION = "h9_image_zero_shot_v1"
IMAGE_RANDOM_SEED = 0
IMAGE_SOURCE_CONFIG = {
    "personal_image_ham": {"true_label": "safe", "source_label": "personal_ham"},
    "personal_image_spam": {"true_label": "spam", "source_label": "personal_spam"},
    "spam_archive_jmlr": {"true_label": "spam", "source_label": "jmlr_spam_archive"},
}


def resolve_notebooks_root_for_images() -> Path:
    folder_names = list(IMAGE_SOURCE_CONFIG)
    candidate_data_dirs = []
    if "DATA_DIR" in globals():
        candidate_data_dirs.append(Path(DATA_DIR))
    candidate_data_dirs.extend(
        [
            Path("/content/drive/MyDrive/GemScan/notebooks/data"),
            Path("notebooks/data"),
            Path("GemScan/notebooks/data"),
        ]
    )
    seen = set()
    for candidate in candidate_data_dirs:
        candidate = candidate.expanduser()
        key = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if all((candidate / folder).exists() for folder in folder_names):
            return candidate.parent
    if "NOTEBOOKS_ROOT" in globals():
        return Path(NOTEBOOKS_ROOT)
    return Path("notebooks")


NOTEBOOKS_ROOT = resolve_notebooks_root_for_images()
DATA_DIR = NOTEBOOKS_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = NOTEBOOKS_ROOT / "_results"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_CORPUS_PATH = PROCESSED_DIR / "h9_image_zero_shot_corpus.csv"
IMAGE_SKIPPED_PATH = PROCESSED_DIR / "h9_image_zero_shot_skipped.csv"
IMAGE_REPORT_PATH = RESULTS_DIR / "h9_image_dataset_report.md"


def stable_image_id(source_folder: str, relative_path: str) -> str:
    digest = hashlib.sha256(f"{IMAGE_DATASET_SCHEMA_VERSION}:{source_folder}:{relative_path}".encode("utf-8", errors="replace")).hexdigest()
    return f"img_{digest[:20]}"


def inspect_image(path: Path) -> dict:
    with Image.open(path) as image:
        image_format = image.format or path.suffix.lower().lstrip(".") or "unknown"
        original_mode = image.mode
        n_frames = getattr(image, "n_frames", 1)
        if image_format.upper() == "GIF" and n_frames:
            image.seek(0)
        image.load()
        width, height = image.size
    return {
        "width": int(width),
        "height": int(height),
        "image_format": str(image_format).lower(),
        "original_mode": original_mode,
        "n_frames": int(n_frames) if n_frames else 1,
        "file_size_bytes": int(path.stat().st_size),
    }


def assign_source_balanced_splits(frame: pd.DataFrame, seed: int = IMAGE_RANDOM_SEED) -> pd.DataFrame:
    split_frames = []
    for source_folder, group in frame.groupby("source_folder", sort=True):
        group = group.sort_values("image_id").reset_index(drop=True)
        if len(group) < 3:
            group["split"] = "test"
            split_frames.append(group)
            continue
        train_idx, temp_idx = train_test_split(group.index, test_size=0.30, random_state=seed, shuffle=True)
        temp = group.loc[temp_idx]
        if len(temp) < 2:
            group["split"] = "train"
            group.loc[temp_idx, "split"] = "test"
            split_frames.append(group)
            continue
        val_idx, test_idx = train_test_split(temp.index, test_size=0.50, random_state=seed, shuffle=True)
        group["split"] = "train"
        group.loc[val_idx, "split"] = "val"
        group.loc[test_idx, "split"] = "test"
        split_frames.append(group)
    return pd.concat(split_frames, ignore_index=True).sort_values(["source_folder", "split", "image_id"]).reset_index(drop=True)


image_rows = []
skipped_rows = []

for source_folder, config in IMAGE_SOURCE_CONFIG.items():
    folder = DATA_DIR / source_folder
    if not folder.exists():
        skipped_rows.append(
            {
                "benchmark_schema_version": IMAGE_DATASET_SCHEMA_VERSION,
                "image_id": stable_image_id(source_folder, source_folder),
                "source_folder": source_folder,
                "relative_path": source_folder,
                "error_type": "missing_source_folder",
                "error_message": f"Missing image source folder: {folder}",
            }
        )
        continue

    for path in sorted(candidate for candidate in folder.iterdir() if candidate.is_file()):
        relative_path = str(path.relative_to(DATA_DIR))
        image_id = stable_image_id(source_folder, relative_path)
        try:
            image_info = inspect_image(path)
        except Exception as exc:
            skipped_rows.append(
                {
                    "benchmark_schema_version": IMAGE_DATASET_SCHEMA_VERSION,
                    "image_id": image_id,
                    "source_folder": source_folder,
                    "relative_path": relative_path,
                    "error_type": exc.__class__.__name__,
                    "error_message": str(exc)[:500],
                }
            )
            continue

        image_rows.append(
            {
                "benchmark_schema_version": IMAGE_DATASET_SCHEMA_VERSION,
                "image_id": image_id,
                "source_folder": source_folder,
                "source_label": config["source_label"],
                "true_label": config["true_label"],
                "true_risk_label": "risk" if config["true_label"] == "spam" else "safe",
                "relative_path": relative_path,
                "split": "unassigned",
                **image_info,
            }
        )

image_corpus = pd.DataFrame(image_rows)
skipped_images = pd.DataFrame(skipped_rows)

if image_corpus.empty:
    raise ValueError(f"No loadable image files found under {DATA_DIR}. Check the H10.5 image source folders.")

image_corpus = image_corpus.drop_duplicates(subset=["image_id"]).reset_index(drop=True)
image_corpus = assign_source_balanced_splits(image_corpus, seed=IMAGE_RANDOM_SEED)
image_corpus.to_csv(IMAGE_CORPUS_PATH, index=False)
skipped_images.to_csv(IMAGE_SKIPPED_PATH, index=False)


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    if frame.empty:
        return "_No rows._"
    markdown_frame = frame.fillna("").astype(str)
    columns = list(markdown_frame.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for _, row in markdown_frame.iterrows():
        values = [str(row[column]).replace("|", "\\|") for column in columns]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

source_summary = image_corpus.groupby(["source_folder", "true_label", "split"]).size().reset_index(name="rows")
format_summary = image_corpus.groupby(["source_folder", "image_format"]).size().reset_index(name="rows")
skipped_summary = (
    skipped_images.groupby(["source_folder", "error_type"]).size().reset_index(name="rows")
    if not skipped_images.empty
    else pd.DataFrame(columns=["source_folder", "error_type", "rows"])
)

report_lines = [
    "# H9 Image Zero-shot Dataset Report",
    "",
    "H10.5 uses this deterministic image corpus for zero-shot multimodal scam/spam image benchmarking.",
    "",
    f"Corpus: `{IMAGE_CORPUS_PATH}`",
    f"Skipped/corrupt files: `{IMAGE_SKIPPED_PATH}`",
    f"Schema version: `{IMAGE_DATASET_SCHEMA_VERSION}`",
    "",
    "## Ground-truth Mapping",
    "",
    "- `personal_image_ham` -> `safe`",
    "- `personal_image_spam` -> `spam`",
    "- `spam_archive_jmlr` -> `spam`",
    "",
    "## Source/Split Shape",
    "",
    dataframe_to_markdown(source_summary),
    "",
    "## Image Formats",
    "",
    dataframe_to_markdown(format_summary),
    "",
    "## Skipped Files",
    "",
    dataframe_to_markdown(skipped_summary),
    "",
]
IMAGE_REPORT_PATH.write_text("\n".join(report_lines))

print("Image corpus:", image_corpus.shape, IMAGE_CORPUS_PATH)
print("Skipped/corrupt files:", skipped_images.shape, IMAGE_SKIPPED_PATH)
print("Dataset report:", IMAGE_REPORT_PATH)
display(source_summary)
display(skipped_summary)
image_corpus.head()

